<a href="https://colab.research.google.com/github/Jaihari-B/Project/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [12]:
from datetime import datetime, timedelta
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

ticker = "AAPL"

today = datetime.now().date()

end_date_for_download = today - timedelta(days=2)
prediction_date = today - timedelta(days=1)

print(f"Downloading data for {ticker} up to {end_date_for_download.strftime('%Y-%m-%d')}")
data = yf.download(ticker, start="2010-01-01", end=end_date_for_download.strftime('%Y-%m-%d'))

price_data = data["Close"].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(price_data)

look_back = 60
X_train, y_train = [], []
for i in range(look_back, len(scaled_data)):
    X_train.append(scaled_data[i-look_back:i, 0])
    y_train.append(scaled_data[i, 0])

X_train, y_train = np.array(X_train), np.array(y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

print(f"Data prepared for training the model to predict {prediction_date.strftime('%Y-%m-%d')}.")

/tmp/ipython-input-3066011087.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2010-01-01", end=end_date_for_download.strftime('%Y-%m-%d'))
[*********************100%***********************]  1 of 1 completed

[[  6.4183836 ]
 [  6.42947912]
 [  6.3272109 ]
 ...
 [255.41000366]
 [258.26998901]
 [256.44000244]]


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(look_back, 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=50, batch_size=32, callbacks=[early_stopping])


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 0.0140
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 64ms/step - loss: 0.0014
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 64ms/step - loss: 0.0011
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - loss: 9.7899e-04
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - loss: 9.9460e-04
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - loss: 7.9337e-04
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 54ms/step - loss: 8.9248e-04
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - loss: 8.6594e-04
Epoch 9/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - loss: 7.2228e-04
Epoch 10/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - loss: 7.6595e-04
Epoch 11/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 7.8853e-04
Epoch 12/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - loss: 6.5033e-04
Epoch 13/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - loss: 7.4446e-04
Epoch 14/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - loss: 7.0787e-04
Epoch

In [8]:
from datetime import datetime, timedelta
import numpy as np

prediction_date = datetime.now().date() - timedelta(days=1)

input_for_yesterday_pred = scaled_data[-look_back:].reshape(1, look_back, 1)
yesterday_predicted_scaled = model.predict(input_for_yesterday_pred)

yesterday_predicted_price = scaler.inverse_transform(yesterday_predicted_scaled)

print(f"Predicted price for {prediction_date.strftime('%Y-%m-%d')}: ${yesterday_predicted_price[0][0]:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
Predicted price for 2026-01-30: $252.12


In [9]:
actual_yesterday_price = 259.48
predicted_yesterday_price = yesterday_predicted_price[0][0]

error = abs(actual_yesterday_price - predicted_yesterday_price)
percentage_error = (error / actual_yesterday_price) * 100

print(f"Actual Price for yesterday ({prediction_date.strftime('%Y-%m-%d')}): ${actual_yesterday_price:.2f}")
print(f"Predicted Price for yesterday ({prediction_date.strftime('%Y-%m-%d')}): ${predicted_yesterday_price:.2f}")
print(f"Prediction Error: ${error:.2f}")
print(f"Percentage Error: {percentage_error:.2f}%")

Actual Price for yesterday (2026-01-30): $259.48
Predicted Price for yesterday (2026-01-30): $252.12
Prediction Error: $7.36
Percentage Error: 2.84%
